# Multi-Agent Stock Trading Comparison (PPO vs DDPG vs SAC vs Ensemble)

This notebook follows the training and testing workflow of `ppo_stock.ipynb`, then compares four RL strategies (`PPO`, `DDPG`, `SAC`, `Ensemble`) against `DJI` benchmark on the same test horizon.

The `Ensemble` strategy averages the actions from the trained `PPO`, `DDPG`, and `SAC` agents at inference time, then runs the shared action inside the same trading environment.

Outputs:
- Trained checkpoints for PPO, DDPG, and SAC
- Test account value CSV files for PPO, DDPG, SAC, and Ensemble
- Comparison charts and metric tables including the DJI benchmark

In [137]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Compatibility patches used in existing notebooks/scripts
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return torch._original_load(*args, **kwargs)

torch.load = patched_torch_load

if not hasattr(yf, '_original_download'):
    yf._original_download = yf.download

def patched_download(*args, **kwargs):
    kwargs.pop('proxy', None)
    kwargs.setdefault('progress', False)
    kwargs.setdefault('auto_adjust', False)
    return yf._original_download(*args, **kwargs)

yf.download = patched_download

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from elegantrl.agents import AgentPPO, AgentDDPG, AgentSAC
from elegantrl.train.run import train_agent
from env_wrapper.ppo_env_wrapper import ElegantFinRLWrapper

try:
    from elegantrl.train.config import Config as Arguments
except ImportError:
    from elegantrl.train.config import Arguments

print('[OK] Imports complete')

[OK] Imports complete


In [138]:
ALL_INDICATORS = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30', 'close_60']

def prepare_data(cache_file='./data/stock_dow30_cache.csv',
                 TRAIN_START_DATE='2010-01-01',
                 TEST_END_DATE='2023-12-30'):
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)

    if os.path.exists(cache_file):
        print(f'[INFO] Loading cache: {cache_file}')
        try:
            df = pd.read_csv(cache_file)
            df = df.drop_duplicates(subset=['date', 'tic'])
            if len(df) > 0:
                print(f'[OK] Cache rows={len(df)}, range={df["date"].min()} -> {df["date"].max()}')
                return df
        except Exception as e:
            print(f'[WARN] Cache load failed: {e}; redownloading')

    print('[STEP] Downloading raw DOW30 data...')
    df = YahooDownloader(
        start_date=TRAIN_START_DATE,
        end_date=TEST_END_DATE,
        ticker_list=DOW_30_TICKER
    ).fetch_data()

    print('[STEP] Adding technical indicators...')
    tech_for_fe = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']

    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=tech_for_fe,
        use_vix=False,
        use_turbulence=False,
        user_defined_feature=False
    )
    df = fe.preprocess_data(df)

    if 'close_30_sma' in df.columns:
        df = df.rename(columns={'close_30_sma': 'close_30'})
    if 'close_60_sma' in df.columns:
        df = df.rename(columns={'close_60_sma': 'close_60'})

    df.to_csv(cache_file, index=False)
    print(f'[OK] Cached to {cache_file}')
    return df

def split_train_test(df,
                     TRAIN_START_DATE='2010-01-07',
                     TRAIN_END_DATE='2023-10-24',
                     TEST_START_DATE1='2023-10-25',
                     TEST_END_DATE1='2023-11-14',
                     TEST_START_DATE2='2023-11-15',
                     TEST_END_DATE2='2023-11-22'):
    df['date'] = pd.to_datetime(df['date'])

    df_train = df[(df['date'] >= TRAIN_START_DATE) & (df['date'] <= TRAIN_END_DATE)].copy()
    df_test1 = df[(df['date'] >= TEST_START_DATE1) & (df['date'] <= TEST_END_DATE1)].copy()
    df_test2 = df[(df['date'] >= TEST_START_DATE2) & (df['date'] <= TEST_END_DATE2)].copy()

    return df_train, df_test1, df_test2

df = prepare_data()
df_train, df_test1, df_test2 = split_train_test(df)

print(f'[INFO] train rows={len(df_train)}, test1 rows={len(df_test1)}, test2 rows={len(df_test2)}')

[INFO] Loading cache: ./data/stock_dow30_cache.csv
[OK] Cache rows=29174, range=2020-01-02 -> 2023-12-29
[INFO] train rows=27840, test1 rows=435, test2 rows=174


In [139]:
def add_day_index(df_part):
    out = df_part.copy()
    dates = sorted(out['date'].unique())
    date_map = {d: i for i, d in enumerate(dates)}
    out['day'] = out['date'].map(date_map)
    out.set_index('day', inplace=True, drop=False)
    return out

df_train = add_day_index(df_train)
df_test1 = add_day_index(df_test1)
df_test2 = add_day_index(df_test2)

stock_dimension = len(df_train['tic'].unique())
state_dim = 1 + stock_dimension * (len(ALL_INDICATORS) + 2) # cash + (每只股票的持仓数量 + 价格 + 技术指标)

BASE_ENV_PARAMS = {
    'env_name': 'FinRL_Stock_Train',
    'stock_dim': stock_dimension,
    'hmax': 100,
    'initial_amount': 1_000_000,
    'num_stock_shares': [0] * stock_dimension,
    'buy_cost_pct': [0.001] * stock_dimension,
    'sell_cost_pct': [0.001] * stock_dimension,
    'reward_scaling': 1e-4,
    'state_space': state_dim,
    'action_space': stock_dimension,
    'tech_indicator_list': ALL_INDICATORS,
    'state_dim': state_dim,
    'action_dim': stock_dimension,
    'if_discrete': False,
    'target_return': 10.0,
}

print(f'[INFO] stock_dimension={stock_dimension}, state_dim={state_dim}')

[INFO] stock_dimension=29, state_dim=291


In [140]:
def setup_args(name, agent_class, env_args, cwd_path, should_train):

    args = Arguments(agent_class=agent_class, env_class=ElegantFinRLWrapper)
    args.env_args = env_args
    args.env_name = env_args['env_name']

    args.state_dim = env_args['state_dim']
    args.action_dim = env_args['action_dim']
    args.if_discrete = env_args['if_discrete']

    if name == "PPO":
        args.net_dims = (128,64)
        args.learning_rate = 2e-4
        args.batch_size = 128
        args.target_step = 2000
        args.repeat_times = 8
        args.break_step = 40000
        

    elif name == "DDPG":

        args.net_dims = (256,128)
        args.learning_rate = 2e-4
        args.batch_size = 256
        args.target_step = 2000
        args.repeat_times = 1
        args.buffer_size = int(1e6)
        args.break_step = 50000

    elif name == "SAC":

        args.net_dims = (256,256)
        args.learning_rate = 3e-4
        args.batch_size = 256
        args.target_step = 1024
        args.repeat_times = 1
        args.buffer_size = int(1e6)
        args.break_step = 2e5

    args.if_remove = should_train
    args.gamma = 0.99
    args.tau = 5e-3

    

    args.worker_num = 1
    args.eval_gap = 2000
    args.save_gap = 2000

    args.if_save = True
    args.if_overwrite_save = True

    args.cwd = cwd_path
    

    return args


def load_trained_agent(agent_class, args):
    agent = agent_class(args.net_dims, args.state_dim, args.action_dim)
    agent.save_or_load_agent(args.cwd, if_save=False)
    agent.act.eval()
    return agent


def predict_action(agent, state):
    s_tensor = torch.as_tensor((state,), dtype=torch.float32, device=agent.device)
    with torch.no_grad():
        action = agent.act(s_tensor).detach().cpu().numpy()[0]

    action = np.nan_to_num(action, nan=0.0, posinf=1.0, neginf=-1.0)
    return np.clip(action, -1.0, 1.0)


def finalize_test_results(name, pred1, pred2, out_dir='./logs/stock'):
    pred1 = pred1.copy()
    pred2 = pred2.copy()
    pred1['date'] = pd.to_datetime(pred1['date'])
    pred2['date'] = pd.to_datetime(pred2['date'])

    merged = pd.concat([pred1, pred2], axis=0, ignore_index=True)
    merged = merged.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)

    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/{name.lower()}_test_results_tot.csv'
    merged.to_csv(out_file, index=False)
    print(f'[OK] Saved {name} test results -> {out_file}')
    return merged


def run_inference(agent_class, args, test_df, env_params):
    test_env_params = dict(env_params)
    test_env_params['df'] = test_df

    env = ElegantFinRLWrapper(**test_env_params)
    agent = load_trained_agent(agent_class, args)

    res = env.reset()
    state = res[0] if isinstance(res, tuple) else res

    done = False
    while not done:
        action = predict_action(agent, state)

        step_res = env.step(action)
        if len(step_res) == 5:
            state, reward, term, trunc, _ = step_res
            done = bool(term or trunc)
        else:
            state, reward, done, _ = step_res

    return env.env.save_asset_memory()


def run_ensemble_inference(agent_specs, test_df, env_params, ensemble_weights):
    test_env_params = dict(env_params)
    test_env_params['df'] = test_df

    env = ElegantFinRLWrapper(**test_env_params)
    agents = [load_trained_agent(agent_class, args) for _, agent_class, args in agent_specs]

    res = env.reset()
    state = res[0] if isinstance(res, tuple) else res

    done = False
    while not done:
        action_stack = np.stack([predict_action(agent, state) for agent in agents], axis=0)
        # apply essemble weights
        ensemble_action = np.sum(action_stack * np.array(ensemble_weights)[:, None], axis=0)
        # ensemble_action = np.mean(action_stack, axis=0)

        step_res = env.step(ensemble_action)
        if len(step_res) == 5:
            state, reward, term, trunc, _ = step_res
            done = bool(term or trunc)
        else:
            state, reward, done, _ = step_res

    return env.env.save_asset_memory()


def build_ensemble_from_member_results(member_results, member_names=('PPO', 'DDPG', 'SAC')):
    aligned = None
    for name in member_names:
        tmp = member_results[name][['date', 'account_value']].copy()
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp.rename(columns={'account_value': name})

        if aligned is None:
            aligned = tmp
        else:
            aligned = aligned.merge(tmp, on='date', how='outer')

    aligned = aligned.sort_values('date').reset_index(drop=True)
    aligned[list(member_names)] = aligned[list(member_names)].ffill().bfill()

    member_returns = aligned[list(member_names)].pct_change().fillna(0.0)
    ensemble_returns = member_returns.mean(axis=1)
    initial_capital = float(aligned[list(member_names)].iloc[0].mean())
    ensemble_curve = initial_capital * (1.0 + ensemble_returns).cumprod()

    return pd.DataFrame({
        'date': aligned['date'],
        'account_value': ensemble_curve,
    })


def train_and_test_one(name, agent_class, train_df, test_df1, test_df2, base_env_params, should_train=True):
    print('\n' + '=' * 80)
    print(f'[TRAIN] {name}')
    print('=' * 80)

    env_params = dict(base_env_params)
    env_params['env_name'] = f'FinRL_{name}_Train'
    env_params['df'] = train_df

    ckpt_dir = f'./checkpoints/stock/{name.lower()}_full_train'
    os.makedirs(ckpt_dir, exist_ok=True)

    args = setup_args(name, agent_class, env_params, ckpt_dir, should_train)
    if should_train:
        train_agent(args)

    print(f'[TEST] {name} inference on test windows')
    pred1 = run_inference(agent_class, args, test_df1, env_params)
    pred2 = run_inference(agent_class, args, test_df2, env_params)
    merged = finalize_test_results(name, pred1, pred2)

    return merged, args

In [224]:
def generate_ensemble_weights(ENSEMBLE_MEMBERS, num_weight=10):
    """ PPO > DDPG >> SAC randomly generate different weights based on this.
     """
    
    base_weights = np.array([0.8, 0.6, 1.0])  # Base weights for PPO, DDPG, SAC
    base_weights = base_weights / base_weights.sum()  # Normalize to sum to 1
    ensemble_agent_weights_list = []
    for _ in range(num_weight):
        random_factors = np.random.uniform(0.5, 1.5, size=base_weights.shape)  # Random factors to add variability
        weights = base_weights * random_factors  # Apply random factors
        weights = weights / weights.sum()  # Normalize to sum to 1
        ensemble_agent_weights_list.append({
            'PPO': weights[0],
            'DDPG': weights[1],
            'SAC': weights[2],
        })
    
    return ensemble_agent_weights_list

In [225]:
def calculate_metrics(df, value_col='account_value'):
    x = df[value_col].dropna()
    daily = x.pct_change().dropna()

    ret = float(x.iloc[-1] / x.iloc[0] - 1.0) if len(x) > 1 else 0.0
    shp = float((252 ** 0.5) * (daily.mean() / daily.std())) if daily.std() != 0 else 0.0

    running_max = x.cummax()
    dd = (x - running_max) / running_max
    mdd = float(dd.min()) if len(dd) else 0.0

    return ret, shp, mdd


In [226]:
# Set to False if you already have result CSVs and only want to re-plot
RUN_TRAINING = True

AGENTS = {
    'PPO': AgentPPO,
    'DDPG': AgentDDPG,
    'SAC': AgentSAC,
}

SHOULD_TRAIN = {
    'PPO': False,
    'DDPG': False,
    'SAC': False,
}

ENSEMBLE_MEMBERS = tuple(AGENTS.keys())

results = {}
trained_agent_specs = {}
best_ensemble_cret = -np.inf

if RUN_TRAINING:
    for name, agent_class in AGENTS.items():
        result_df, args = train_and_test_one(
            name=name,
            agent_class=agent_class,
            train_df=df_train,
            test_df1=df_test1,
            test_df2=df_test2,
            base_env_params=BASE_ENV_PARAMS,
            should_train=SHOULD_TRAIN.get(name, False)
        )
        results[name] = result_df
        trained_agent_specs[name] = (agent_class, args)
    # eseemble
    ensemble_agent_weights_list = generate_ensemble_weights(ENSEMBLE_MEMBERS, num_weight=100)
    for i, ensemble_agent_weights in enumerate(ensemble_agent_weights_list):
        ensemble_specs = [
            (name, trained_agent_specs[name][0], trained_agent_specs[name][1])
            for name in ENSEMBLE_MEMBERS
        ]
        print('[TEST] Ensemble inference on test windows')
        # weight should be corresponding to the order of ensemble_specs
        ensemble_agent_weights = [ensemble_agent_weights[name] for name in ENSEMBLE_MEMBERS]
        # normalize weights
        total_weight = sum(ensemble_agent_weights)
        ensemble_agent_weights = [w / total_weight for w in ensemble_agent_weights]

        ensemble_pred1 = run_ensemble_inference(ensemble_specs, df_test1, BASE_ENV_PARAMS, ensemble_agent_weights)
        ensemble_pred2 = run_ensemble_inference(ensemble_specs, df_test2, BASE_ENV_PARAMS, ensemble_agent_weights)
        r, s, m = calculate_metrics(ensemble_pred1, 'account_value')
        if r > best_ensemble_cret:
            best_ensemble_cret = r
            results['Ensemble'] = finalize_test_results('Ensemble', ensemble_pred1, ensemble_pred2)
else:
    for name in AGENTS.keys():
        f = f'./logs/stock/{name.lower()}_test_results_tot.csv'
        if not os.path.exists(f):
            raise FileNotFoundError(f'Cannot find {f}, please train first or set RUN_TRAINING=True')
        tmp = pd.read_csv(f)
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
        results[name] = tmp

    ensemble_file = './logs/stock/ensemble_test_results_tot.csv'
    if os.path.exists(ensemble_file):
        ensemble_df = pd.read_csv(ensemble_file)
        ensemble_df['date'] = pd.to_datetime(ensemble_df['date'])
        ensemble_df = ensemble_df.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
        results['Ensemble'] = ensemble_df
        print(f'[INFO] Loaded existing Ensemble results from {ensemble_file}')
    else:
        print('[INFO] Ensemble result CSV not found, building one from PPO/DDPG/SAC account curves')
        ensemble_df = build_ensemble_from_member_results(results, ENSEMBLE_MEMBERS)
        results['Ensemble'] = finalize_test_results('Ensemble', ensemble_df.iloc[:0].copy(), ensemble_df)

print('[OK] Result data prepared for PPO/DDPG/SAC/Ensemble')


[TRAIN] PPO
[TEST] PPO inference on test windows
[OK] Saved PPO test results -> ./logs/stock/ppo_test_results_tot.csv

[TRAIN] DDPG
[TEST] DDPG inference on test windows
[OK] Saved DDPG test results -> ./logs/stock/ddpg_test_results_tot.csv

[TRAIN] SAC
[TEST] SAC inference on test windows
[OK] Saved SAC test results -> ./logs/stock/sac_test_results_tot.csv
[TEST] Ensemble inference on test windows
[OK] Saved Ensemble test results -> ./logs/stock/ensemble_test_results_tot.csv
[TEST] Ensemble inference on test windows
[OK] Saved Ensemble test results -> ./logs/stock/ensemble_test_results_tot.csv
[TEST] Ensemble inference on test windows
[OK] Saved Ensemble test results -> ./logs/stock/ensemble_test_results_tot.csv
[TEST] Ensemble inference on test windows
[TEST] Ensemble inference on test windows
[TEST] Ensemble inference on test windows
[TEST] Ensemble inference on test windows
[TEST] Ensemble inference on test windows
[TEST] Ensemble inference on test windows
[OK] Saved Ensemble test

In [227]:
def build_dji_curve(reference_df, initial_capital):
    start_date = reference_df['date'].min().strftime('%Y-%m-%d')
    end_date = (reference_df['date'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    bench = yf.download('^DJI', start=start_date, end=end_date, progress=False)
    if isinstance(bench.columns, pd.MultiIndex):
        bench.columns = bench.columns.get_level_values(0)

    bench = bench.reset_index()
    if 'Date' not in bench.columns and 'index' in bench.columns:
        bench = bench.rename(columns={'index': 'Date'})

    bench['Date'] = pd.to_datetime(bench['Date'])

    merged = pd.merge(
        reference_df[['date']].copy(),
        bench[['Date', 'Close']],
        left_on='date',
        right_on='Date',
        how='left'
    )
    merged['Close'] = merged['Close'].ffill().bfill()
    merged['account_value'] = (merged['Close'] / merged['Close'].iloc[0]) * initial_capital

    return merged[['date', 'account_value']]


def plot_comparison_for_period(start_date, end_date, tag):
    model_names = ['Ensemble', 'SAC', 'DDPG', 'PPO']
    period_results = {}

    for name in model_names:
        tmp = results[name].copy()
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp[(tmp['date'] >= start_date) & (tmp['date'] <= end_date)].copy()
        tmp = tmp.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
        period_results[name] = tmp

    base = period_results['Ensemble'][['date']].copy()
    for name in model_names:
        period_results[name] = pd.merge(base, period_results[name][['date', 'account_value']], on='date', how='left')
        period_results[name]['account_value'] = period_results[name]['account_value'].ffill().bfill()

    initial_capital = float(period_results['Ensemble']['account_value'].iloc[0])
    period_results['DJI'] = build_dji_curve(period_results['Ensemble'], initial_capital)

    metrics = {}
    for name in model_names + ['DJI']:
        r, s, m = calculate_metrics(period_results[name], 'account_value')
        metrics[name] = {'ret': r, 'shp': s, 'mdd': m}

    plt.style.use('seaborn-v0_8-darkgrid')

    fig = plt.figure(figsize=(14, 8))
    ax = plt.gca()

    # ===== 统一颜色 =====
    colors = {
        'Ensemble': 'darkviolet',
        'SAC': 'crimson',
        'DDPG': 'dodgerblue',
        'PPO': 'forestgreen'
    }

    # ===== 画策略曲线 (统一 linewidth / alpha / zorder) =====  
    for algo in ['SAC', 'DDPG', 'PPO', 'Ensemble']:

        lw = 3.5 if algo == 'Ensemble' else 1.5
        alpha = 1.0 if algo == 'Ensemble' else 0.7
        zorder = 5 if algo == 'Ensemble' else 3

        ax.plot(
            period_results[algo]['date'],
            period_results[algo]['account_value'],
            label=f'{algo} Portfolio',
            color=colors[algo],
            linewidth=lw,
            alpha=alpha,
            zorder=zorder
        )

    # ===== Benchmark =====
    ax.plot(
        period_results['DJI']['date'],
        period_results['DJI']['account_value'],
        label='Dow Jones (Benchmark)',
        color='slategray',
        linewidth=2.0,
        linestyle='--',
        zorder=2
    )

    # ===== 标题 =====
    # ax.set_title(
    #     f'DRL Trading Comparison: Single Agents vs. Ensemble Agent ({tag})',
    #     fontsize=18,
    #     fontweight='bold',
    #     pad=15
    # )

    ax.set_xlabel('Date', fontsize=14)
    ax.set_ylabel('Portfolio Value ($)', fontsize=16)
    ax.tick_params(axis='both', labelsize=14)
    plt.xticks(rotation=45)

    # ===== 指标文本 (严格对齐) =====
    metrics_text = [
        f"{'SAC':<8} Ret: {metrics['SAC']['ret']*100:>+7.2f}% | Shp: {metrics['SAC']['shp']:>5.2f} | MDD: {metrics['SAC']['mdd']*100:>7.2f}%",
        f"{'DDPG':<8} Ret: {metrics['DDPG']['ret']*100:>+7.2f}% | Shp: {metrics['DDPG']['shp']:>5.2f} | MDD: {metrics['DDPG']['mdd']*100:>7.2f}%",
        f"{'PPO':<8} Ret: {metrics['PPO']['ret']*100:>+7.2f}% | Shp: {metrics['PPO']['shp']:>5.2f} | MDD: {metrics['PPO']['mdd']*100:>7.2f}%",
        f"{'Ensemble':<8} Ret: {metrics['Ensemble']['ret']*100:>+7.2f}% | Shp: {metrics['Ensemble']['shp']:>5.2f} | MDD: {metrics['Ensemble']['mdd']*100:>7.2f}%",
        f"{'DJI':<8} Ret: {metrics['DJI']['ret']*100:>+7.2f}% | Shp: {metrics['DJI']['shp']:>5.2f} | MDD: {metrics['DJI']['mdd']*100:>7.2f}%"
    ]

    textstr = "\n".join(metrics_text)

    props = dict(
        boxstyle='round,pad=0.5',
        facecolor='#f8f9fa',
        edgecolor='gray',
        alpha=0.9
    )

    ax.text(
        0.02, 0.96,
        textstr,
        transform=plt.gca().transAxes,
        fontsize=18,
        verticalalignment='top',
        bbox=props,
        family='monospace',
        # fontweight='bold',
        color='#333333'
    )

    # ===== Legend (左下角) =====
    ax.legend(
        loc='upper left',
        bbox_to_anchor=(0.02, 0.72),
        fontsize=18,
        framealpha=0.9
    )   


    plt.tight_layout()

    os.makedirs('./logs/stock', exist_ok=True)

    out_fig = f'./logs/stock/rl_trading_comparison_{tag.lower().replace(" ", "_")}.png'
    plt.savefig(out_fig, dpi=300)

    print(f'[OK] Saved figure -> {out_fig}')

    plt.show()


    return pd.DataFrame([
        {'Strategy': name, 'Return(%)': metrics[name]['ret'] * 100, 'Sharpe': metrics[name]['shp'], 'MaxDrawdown(%)': metrics[name]['mdd'] * 100}
        for name in model_names + ['DJI']
    ])


# Plot test set 1 and test set 2 separately based on split ranges
test1_start = pd.to_datetime(df_test1['date']).min()
test1_end = pd.to_datetime(df_test1['date']).max()
test2_start = pd.to_datetime(df_test2['date']).min()
test2_end = pd.to_datetime(df_test2['date']).max()

summary_test1 = plot_comparison_for_period(test1_start, test1_end, 'Test Set 1')
summary_test2 = plot_comparison_for_period(test2_start, test2_end, 'Test Set 2')

print('\n[SUMMARY] Test Set 1')
display(summary_test1)
print('\n[SUMMARY] Test Set 2')
display(summary_test2)

[OK] Saved figure -> ./logs/stock/rl_trading_comparison_test_set_1.png
[OK] Saved figure -> ./logs/stock/rl_trading_comparison_test_set_2.png

[SUMMARY] Test Set 1


,Strategy,Return(%),Sharpe,MaxDrawdown(%)
0,Ensemble,6.644718,8.862873,-0.796877
1,SAC,6.444301,11.141142,-0.659439
2,DDPG,6.659279,9.735642,-0.807704
3,PPO,7.320131,9.290777,-0.963161
4,DJI,5.423699,6.890874,-1.871719



[SUMMARY] Test Set 2


,Strategy,Return(%),Sharpe,MaxDrawdown(%)
0,Ensemble,0.463177,5.067739,-0.329034
1,SAC,1.054428,9.762004,-0.099187
2,DDPG,0.690624,7.515482,-0.282901
3,PPO,0.596383,5.343358,-0.428932
4,DJI,0.805403,6.983815,-0.178515
